In [ ]:
import json
from pathlib import Path
from PIL import Image
import numpy as np
from typing import Tuple, Dict, List


# ============================================================
# Utilities: WCAG Contrast (Phase-2b)
# ============================================================

def _srgb_channel_to_linear(c: float) -> float:
    c = c / 255.0
    return c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4


def relative_luminance(rgb: Tuple[int, int, int]) -> float:
    r, g, b = rgb
    R = _srgb_channel_to_linear(r)
    G = _srgb_channel_to_linear(g)
    B = _srgb_channel_to_linear(b)
    return 0.2126 * R + 0.7152 * G + 0.0722 * B


def contrast_ratio(fg_rgb: Tuple[int, int, int],
                   bg_rgb: Tuple[int, int, int]) -> float:
    L1 = relative_luminance(fg_rgb)
    L2 = relative_luminance(bg_rgb)
    lighter = max(L1, L2)
    darker = min(L1, L2)
    return (lighter + 0.05) / (darker + 0.05)


# ============================================================
# Phase-2a: Minimal In-Memory Sampling
# ============================================================

def sample_background_from_pixels(image: Image.Image) -> dict:
    """
    Samples effective background color from image corners.
    """
    img = image.convert("RGB")
    w, h = img.size
    px = img.load()

    corners = [
        px[2, 2],
        px[w - 3, 2],
        px[2, h - 3],
        px[w - 3, h - 3]
    ]

    avg_rgb = tuple(
        int(sum(c[i] for c in corners) / len(corners))
        for i in range(3)
    )

    return {
        "effective_background_rgb": list(avg_rgb),
        "background_source": "page",
        "overlay_detected": False
    }


def sample_text_regions(image: Image.Image,
                        background_rgb: Tuple[int, int, int]) -> list:
    """
    Minimal heuristic text sampling.
    """
    img = image.convert("RGB")
    arr = np.array(img)
    bg = np.array(background_rgb)

    diff = np.linalg.norm(arr - bg, axis=2)
    text_mask = diff > 40  # conservative threshold

    regions = []
    sampled = 0
    step = max(5, min(img.size) // 200)

    for y in range(0, img.size[1], step):
        for x in range(0, img.size[0], step):
            if text_mask[y, x]:
                regions.append({
                    "region_id": f"text_{sampled:03d}",
                    "bbox_px": [x, y, 1, 1],
                    "foreground_rgb": arr[y, x].tolist(),
                    "font_px_estimate": None,
                    "rotated": False,
                    "confidence": "low"
                })
                sampled += 1
            if sampled >= 50:
                break
        if sampled >= 50:
            break

    return regions


def sample_figure_visuals(image_path: str,
                          figure_record: dict,
                          rendering_assumptions: dict) -> dict:
    """
    Phase-2a sampling orchestrator (in memory).
    """
    image = Image.open(image_path)

    background = resolve_effective_background(
        image=image,
        figure_record=figure_record,
        rendering_assumptions=rendering_assumptions
    )

    bg_rgb = tuple(background["effective_background_rgb"])
    text_regions = sample_text_regions(image, bg_rgb)

    return {
        "figure_id": figure_record["figure_id"],
        "sampling_version": "phase2a_minimal_v1.0",
        "background": background,
        "text_regions": text_regions,
        "graphic_regions": [],
        "legend_regions": [],
        "notes": [
            "Minimal heuristic sampling",
            "No OCR",
            "Bounding boxes approximate"
        ]
    }


# ============================================================
# Phase-2b: Text Contrast Metrics
# ============================================================

def compute_text_contrast_metrics(
    text_regions: List[dict],
    background_rgb: Tuple[int, int, int],
    wcag_threshold: float = 4.5,
    borderline_delta: float = 0.3
) -> dict:

    ratios = []
    failing = []
    borderline = []

    for region in text_regions:
        fg_rgb = tuple(region["foreground_rgb"])
        ratio = contrast_ratio(fg_rgb, background_rgb)
        ratios.append(ratio)

        if ratio < wcag_threshold:
            failing.append({
                "region_id": region["region_id"],
                "contrast_ratio": round(ratio, 2)
            })
        elif wcag_threshold <= ratio < wcag_threshold + borderline_delta:
            borderline.append({
                "region_id": region["region_id"],
                "contrast_ratio": round(ratio, 2)
            })

    return {
        "min_contrast_ratio": round(min(ratios), 2) if ratios else None,
        "wcag_threshold": wcag_threshold,
        "num_text_elements_sampled": len(text_regions),
        "failing_elements": failing,
        "borderline_elements": borderline
    }


def compute_text_contrast_flags(metrics: dict) -> dict:
    return {
        "text_contrast_meets_wcag": (
            metrics["min_contrast_ratio"] is not None and
            metrics["min_contrast_ratio"] >= metrics["wcag_threshold"]
        ),
        "text_contrast_borderline": bool(metrics["borderline_elements"]),
        "text_contrast_has_failures": bool(metrics["failing_elements"])
    }


def compute_figure_metrics(sampling_record: dict,
                           rendering_assumptions: dict) -> dict:
    bg_rgb = tuple(
        sampling_record["background"]["effective_background_rgb"]
    )

    text_metrics = compute_text_contrast_metrics(
        sampling_record["text_regions"],
        bg_rgb
    )

    flags = compute_text_contrast_flags(text_metrics)

    return {
        "metrics_version": "phase2b_v1.0",
        "measurements": {
            "background": sampling_record["background"],
            "text_contrast": text_metrics
        },
        "metric_flags": flags
    }

def resolve_effective_background(
    image: Image.Image,
    figure_record: dict,
    rendering_assumptions: dict
) -> dict:
    """
    Determines the effective background color for ADA analysis,
    using source-type–aware Word rendering rules.
    """

    source_type = figure_record["source"]["source_type"]

    # --- Case A: Word-native charts ---
    if source_type == "word_chart":
        return {
            "effective_background_rgb": rendering_assumptions["page_background_rgb"],
            "source": "word_page",
            "reason": "word_native_chart"
        }

    # --- Case B: Inserted raster images ---
    if source_type == "image":
        img = image.convert("RGBA")

        # Check for transparency
        alpha = img.split()[-1]
        has_transparency = alpha.getextrema()[0] < 255

        if has_transparency:
            return {
                "effective_background_rgb": rendering_assumptions["page_background_rgb"],
                "source": "word_page",
                "reason": "transparent_inserted_image"
            }

        # Fully opaque → pixel background is meaningful
        sampled = sample_background_from_pixels(img)

        return {
            "effective_background_rgb": sampled["effective_background_rgb"],
            "source": "image_pixels",
            "reason": "opaque_inserted_image"
        }

    # --- Fallback (defensive) ---
    return {
        "effective_background_rgb": rendering_assumptions["page_background_rgb"],
        "source": "word_page",
        "reason": "fallback_default"
    }

# ============================================================
# Phase-2a helpers: Graphic metadata (NEW)
# ============================================================

def infer_graphic_role(fig_record: dict) -> str:
    """
    Deterministically infer the graphic role.
    Conservative defaults; no judgment.
    """
    src = fig_record.get("source", {})
    src_type = src.get("source_type", "")

    if src_type == "word_chart":
        return "line"   # default; refined later if needed
    return "graphic"


def extract_representative_color(image: Image.Image,
                                 bbox_px: Tuple[int, int, int, int]) -> Tuple[int, int, int]:
    """
    Representative color = median RGB within bbox.
    """
    img = image.convert("RGB")
    x1, y1, x2, y2 = bbox_px
    crop = img.crop((x1, y1, x2, y2))
    arr = np.asarray(crop).reshape(-1, 3)
    return tuple(int(v) for v in np.median(arr, axis=0))


def extract_graphic_style(fig_record: dict) -> dict:
    """
    Purely descriptive; no thresholds.
    """
    return {
        "stroke_width_px": 2.0,      # safe default; refined later
        "has_pattern": False,
        "pattern_type": None
    }


# ============================================================
# Map Phase-2 to Schema
# ============================================================

def map_phase2_to_schema(phase2_current: dict) -> dict:
    """
    Convert existing Phase-2 in-memory structure into
    schema-compliant Phase-2 output.
    No measurements are changed.
    """
    document_id = phase2_current.get(
        "document_id",
        phase2_current.get("document", {}).get("filename")
    )

    # rendering_assumptions = (
    #     phase2_current.get("rendering_assumptions")
    #     or phase2_current.get("manifest", {}).get("rendering_assumptions")
    #     or phase2_current.get("document", {}).get("rendering_assumptions")
    # )

    
    phase2_schema = {
        "phase": "phase_2",
        "phase_version": "2.0",
        "document_id": document_id,
        "rendering_assumptions": phase2_current['rendering_assumptions'],
        "figures": []
    }

    for fig in phase2_current["figures"]:
        essential = []
        decorative = []
        ambiguous = []
        if "metrics" not in fig:
            raise KeyError(f"{fig['figure_id']}: missing metrics")

        if "text_contrast" not in fig["metrics"]:
            raise KeyError(f"{fig['figure_id']}: missing text_contrast metrics")

        regions = fig["metrics"]["text_contrast"].get("regions", [])

        for r in regions:
        # --- classify regions ---
            region_entry = {
                "region_id": r["region_id"],
                "region_role": r["region_class"],
                "region_type": r.get("region_type"),
                "geometry": {
                    "bbox_px": r.get("bbox_px")
                },
                "pixel_metrics": {
                    "pixels_analyzed": r["pixels_analyzed"],
                    "pixels_below_threshold": r["pixels_below_threshold"],
                    "percent_below_threshold": r["percent_below_threshold"]
                },
                "contrast_metrics": {
                    "min_contrast": r["min_contrast"],
                    "p05_contrast": r["p05_contrast"]
                },
                "notes": r.get("notes", [])
            }

            if r["region_class"] == "essential":
                essential.append(region_entry)
            elif r["region_class"] == "decorative":
                decorative.append(region_entry)
            else:
                ambiguous.append(region_entry)

        # --- build summaries (NON-authoritative) ---
        essential_pixels_below = sum(
            r["pixel_metrics"]["pixels_below_threshold"] for r in essential
        )

        decorative_pixels_below = sum(
            r["pixel_metrics"]["pixels_below_threshold"] for r in decorative
        )

        figure_out = {
            "figure_id": fig["figure_id"],
            "authoritative_evidence": {
                "essential_regions": essential,
                "decorative_regions": decorative,
                "ambiguous_regions": ambiguous
            },
            "non_authoritative_summaries": {
                "essential_regions_overview": {
                    "regions_analyzed": len(essential),
                    "regions_with_any_failures": sum(
                        1 for r in essential
                        if r["pixel_metrics"]["pixels_below_threshold"] > 0
                    ),
                    "total_pixels_below_threshold": essential_pixels_below
                },
                "decorative_regions_overview": {
                    "regions_analyzed": len(decorative),
                    "total_pixels_below_threshold": decorative_pixels_below
                }
            }
        }

        phase2_schema["figures"].append(figure_out)

    return phase2_schema

# ============================================================
# Unified Phase-2 Driver
# ============================================================

def run_phase2(manifest_path: str,
               output_manifest_path: str,
               output_schema_path: str) -> None:
    """
    Unified Phase-2 driver:
      - reads Phase-0 manifest
      - runs Phase-2a + Phase-2b in memory
      - writes Phase-2 manifest
    """

    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    manifest["phase"] = "phase2"
    manifest["phase2_notes"] = [
        "Phase-2a sampling and Phase-2b metrics executed in-memory",
        "Only text contrast metrics implemented"
    ]

    rendering_assumptions = manifest["rendering_assumptions"]

    for fig in manifest["figures"]:
        image_path = fig["rendered_view"]["image_path"]
        sampling = sample_figure_visuals(
            image_path=image_path,
            figure_record=fig,
            rendering_assumptions=rendering_assumptions
        )

        
        assert image_path, f"{fig['figure_id']}: missing rendered image path"

        # Phase-2a
        sampling = sample_figure_visuals(
            image_path=image_path,
            figure_record=fig,
            rendering_assumptions=rendering_assumptions
        )

        # Phase-2b
        metrics = compute_figure_metrics(
            sampling_record=sampling,
            rendering_assumptions=fig.get("rendering_assumptions", {})
        )

        # Attach results to manifest
        fig["sampling"] = sampling
        fig["metrics"] = metrics

# Structural normalization (this step)
    phase2_schema_ready = map_phase2_to_schema(manifest)


    with open(output_manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    with open(output_schema_path, "w", encoding="utf-8") as fo:
        json.dump(phase2_schema_ready, fo, indent=2)

    print(f"Phase-2 complete. Wrote: {output_manifest_path}")


# ============================================================
# Entry Point
# ============================================================

if __name__ == "__main__":
    run_phase2(
        manifest_path="output/figure_manifest.json",
        output_manifest_path="output/manifest_phase2.json",
        output_schema_path="output/manifest_phase2_schema.json"
    )
    

KeyError: 'Figure_01: missing text_contrast metrics'